# Patch Tuesday Analysis

Pull Microsoft security updates directly from the MSRC API and analyze
exploitation status, severity trends, and impact types across months.
The MSRC API requires no authentication.

In [1]:
from vulnflow import VulnDataset

ds = VulnDataset.from_msrc(start="2025-01", end="2025-04")
print(f"{len(ds):,} CVEs across Jan-Apr 2025")

520 CVEs across Jan-Apr 2025


In [2]:
from vulnflow.transforms.enrichers.epss import EPSSEnricher
from vulnflow.transforms.flatten.cvss import CVSSFlattenTransform
from vulnflow.transforms.flatten.epss import EPSSFlattenTransform
from vulnflow.transforms.flatten.msrc import MSRCFlattenTransform

ds.transform([
    CVSSFlattenTransform(),
    MSRCFlattenTransform(),
    EPSSEnricher(),
    EPSSFlattenTransform(),
])

df = ds.to_pandas()
print(f"{len(df.columns)} columns after enrichment")
df.head(3)

Enriching & transforming:   0%|          | 0/4 [00:00<?, ?transforms/s]

35 columns after enrichment


,id,cve_id,description,nvd_published_at,nvd_modified_at,nvd_status,cvss_version,cvss_base_score,cvss_base_severity,cvss_source,...,epss_bucket,capecs,stride_threats,tags,metadata,msrc_exploited,msrc_publicly_disclosed,msrc_impact,msrc_max_severity,msrc_month_id
0,d3baf404-d4dd-4fed-ab2c-5eb1682effe3,CVE-2024-50338,NaN,NaT,NaT,None,3.1,7.4,HIGH,microsoft,...,very_low,[],[],{},"{""msrc"":{""month_id"":""2025-Jan"",""title"":""GitHub...",True,True,Information Disclosure,Important,2025-Jan
1,a08c6608-f34e-4230-928d-6e4b43fa0b2a,CVE-2025-21411,NaN,NaT,NaT,None,3.1,8.8,HIGH,microsoft,...,low,[],[],{},"{""msrc"":{""month_id"":""2025-Jan"",""title"":""Window...",True,True,Remote Code Execution,Important,2025-Jan
2,ff9eff19-d6f6-42c8-9e9c-b275ac09b993,CVE-2025-21413,NaN,NaT,NaT,None,3.1,8.8,HIGH,microsoft,...,low,[],[],{},"{""msrc"":{""month_id"":""2025-Jan"",""title"":""Window...",True,True,Remote Code Execution,Important,2025-Jan


## Actively Exploited CVEs

These are flagged "Exploitation Detected" by Microsoft — the highest-priority items.

In [3]:
exploited = df[df["msrc_exploited"] == True].sort_values("cvss_base_score", ascending=False)
print(f"{len(exploited)} actively exploited CVEs")
exploited[["cve_id", "msrc_month_id", "cvss_base_score", "msrc_max_severity",
           "msrc_impact", "epss_probability", "description"]]

432 actively exploited CVEs


,cve_id,msrc_month_id,cvss_base_score,msrc_max_severity,msrc_impact,epss_probability,description
501,CVE-2025-30390,2025-Apr,9.9,Critical,Elevation of Privilege,0.00491,<p>Improper authorization in Azure allows an a...
102,CVE-2025-21415,2025-Jan,9.9,Critical,Elevation of Privilege,0.02918,<p>Authentication bypass by spoofing in Azure ...
502,CVE-2025-30392,2025-Apr,9.8,Critical,Elevation of Privilege,0.00559,<p>Improper authorization in Azure Bot Framewo...
158,CVE-2025-21311,2025-Jan,9.8,Critical,Elevation of Privilege,0.06844,NaN
131,CVE-2025-21307,2025-Jan,9.8,Critical,Remote Code Execution,0.12391,NaN
...,...,...,...,...,...,...,...
4,CVE-2025-21210,2025-Jan,4.2,Important,Information Disclosure,0.00172,NaN
233,CVE-2025-21337,2025-Feb,3.3,Important,Elevation of Privilege,0.00142,NaN
134,CVE-2025-21312,2025-Jan,2.4,Important,Information Disclosure,0.00298,NaN
209,CVE-2023-32002,2025-Feb,None,Important,NaN,0.00045,NaN


## Monthly Severity Breakdown

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

severity_order = ["Critical", "Important", "Moderate", "Low"]
colors = {"Critical": "#9c27b0", "Important": "#f44336", "Moderate": "#ff9800", "Low": "#4caf50"}

pivot = (
    df.groupby(["msrc_month_id", "msrc_max_severity"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=severity_order)
    .fillna(0)
    .astype(int)
)

fig, ax = plt.subplots(figsize=(8, 4))
pivot.plot.bar(ax=ax, stacked=True, color=[colors[s] for s in pivot.columns])
ax.set_title("Patch Tuesday: Severity by Month")
ax.set_ylabel("CVE Count")
ax.set_xlabel("")
ax.legend(title="Severity")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## Impact Type Distribution

What kinds of vulnerabilities is Microsoft patching?

In [ ]:
impact_counts = df["msrc_impact"].value_counts()

fig, ax = plt.subplots(figsize=(7, 4))
impact_counts.plot.barh(ax=ax, color="steelblue")
ax.set_title("CVEs by Impact Type (Jan-Apr 2025)")
ax.set_xlabel("CVE Count")
ax.set_ylabel("")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## Exploitation Risk Matrix

Cross-tabulate exploitation status against impact type to see where
active exploitation concentrates.

In [ ]:
risk = pd.crosstab(
    df["msrc_impact"],
    df["msrc_exploited"].map({True: "Exploited", False: "Not Exploited"}),
    margins=True,
)
risk.sort_values("All", ascending=False)

## EPSS vs CVSS — Microsoft Vulnerabilities

Quadrant lines at CVSS = 7 and EPSS = 0.1. Upper-right = high severity
and high exploitation probability. Red dots are actively exploited.

In [ ]:
plot_df = df.dropna(subset=["cvss_base_score", "epss_probability"])

fig, ax = plt.subplots(figsize=(8, 5))

safe = plot_df[plot_df["msrc_exploited"] != True]
expl = plot_df[plot_df["msrc_exploited"] == True]

ax.scatter(safe["cvss_base_score"], safe["epss_probability"],
           alpha=0.4, s=15, color="steelblue", label="Not Exploited")
ax.scatter(expl["cvss_base_score"], expl["epss_probability"],
           alpha=0.9, s=40, color="#f44336", label="Exploited", zorder=5)

ax.axvline(7, color="grey", linestyle="--", linewidth=0.8)
ax.axhline(0.1, color="grey", linestyle="--", linewidth=0.8)
ax.set_xlabel("CVSS Base Score")
ax.set_ylabel("EPSS Probability")
ax.set_title("EPSS vs CVSS — MSRC Patch Tuesday")
ax.legend()
plt.tight_layout()
plt.show()

## Monthly Trend Summary

In [ ]:
summary = (
    df.groupby("msrc_month_id")
    .agg(
        total=("cve_id", "count"),
        critical=("msrc_max_severity", lambda x: (x == "Critical").sum()),
        exploited=("msrc_exploited", lambda x: (x == True).sum()),
        disclosed=("msrc_publicly_disclosed", lambda x: (x == True).sum()),
        avg_cvss=("cvss_base_score", "mean"),
    )
    .round(1)
)
summary

## Save

Write to Parquet for reuse without re-downloading.

In [ ]:
ds.to_parquet("../data/msrc-2025-q1.parquet")
print("Saved to ../data/msrc-2025-q1.parquet")